In [ ]:
!pip install deepxde

In [2]:
import deepxde as dde
import numpy as np
import matplotlib.pyplot as plt

dde.config.disable_xla_jit()  

Using backend: tensorflow
Other supported backends: tensorflow.compat.v1, pytorch, jax, paddle.
paddle supports more examples now and is recommended.
Enable just-in-time compilation with XLA.

Disable just-in-time compilation with XLA.


## Defining The PDE Equation
- `X` and `Y` Input
- `v` Branch Network Input

In [3]:
STARTING_POINT = 0
ENDING_POINT = 5

In [4]:
def burgers_pde(x, y, v):
    dy_t = dde.grad.jacobian(y, x, j=1)
    dy_x = dde.grad.jacobian(y, x, j=0)
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_t + y * dy_x - (0.01/np.pi) * dy_xx

## Domain Interval
- *Time interval* (`0` - `1`)
- *Space Interval* (`-1` - `1`)

In [5]:
space = dde.geometry.Interval(STARTING_POINT, ENDING_POINT)
time = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(space, time)

## Boundary Condition

In [6]:
def boundery_value(x):
    return 0

def is_boundary(x, on_boundary):
    return on_boundary

bc = dde.DirichletBC(geomtime, boundery_value, is_boundary)

def ic_func(x):
    return -np.sin(np.pi * x[:, 0:1])

def is_initial(x, _):
    return np.isclose(x[:, 1], 0)

ic = dde.IC(geomtime, ic_func, is_initial)

## Sensor Point and Space

In [7]:
sensor_x = np.linspace(STARTING_POINT, ENDING_POINT, 100)[:, None]
space = dde.data.GRF((ENDING_POINT - STARTING_POINT), length_scale=0.2, N=1000, interp="cubic")

## Data Generation

In [8]:
pde_data = dde.data.TimePDE(
    geomtime,
    burgers_pde,
    [bc],
    num_domain=200,
    num_boundary=40,
    num_initial=40,
)

data = dde.data.PDEOperatorCartesianProd(
    pde_data,                    # pde        → TimePDE object
    space,                       # function_space → GRF
    sensor_x,                    # evaluation_points → shape (100, 1)
    1000,                        # num_function → 1000 ICs
    function_variables=[0],      # IC depends only on x (index 0), not t (index 1)
    num_test=100,                 # optional test functions
)

In [ ]:
p = 64
net = dde.nn.DeepONetCartesianProd(
    [100, 40, 40, p],
    [2, 40, 40, p],
    "tanh",
    "Glorot normal",
)

# Build Keras variables before DeepXDE wraps the model in tf.function.
_ = net((
    np.zeros((1, 100), dtype=np.float32),
    np.zeros((1, 2), dtype=np.float32),
))

# Step 8: Train
model = dde.Model(data, net)
model.compile("adam", lr=1e-3)

losshistory, train_state = model.train(iterations=1000, display_every=1) 

Compiling model...
'compile' took 0.005801 s

Training model...



In [ ]:
X = np.linspace(STARTING_POINT, ENDING_POINT, 256)
T = np.linspace(0, 1, 100)
XX, TT = np.meshgrid(X, T)

# Branch input: the specific IC you want to query
# Example: u(x,0) = -sin(πx) evaluated at sensor points
sensor_points = sensor_x.flatten()             # shape (100,)
ic_values = -np.sin(np.pi * sensor_points)     # shape (100,)
branch_input = ic_values[np.newaxis, :]         # shape (1, 100) — one IC

# Trunk input: all (x,t) query points
trunk_input = np.column_stack([XX.ravel(), TT.ravel()])  # shape (25600, 2)

# Correct DeepONet predict
u_pred = model.predict((branch_input, trunk_input))      # tuple!
u_pred = u_pred.reshape(100, 256)

plt.figure(figsize=(10, 5))
plt.contourf(XX, TT, u_pred, levels=100, cmap="RdBu_r")
plt.colorbar(label="u(x, t)")
plt.xlabel("x")
plt.ylabel("t")
plt.title("Burgers' Equation — DeepONet Solution")
plt.show()

In [ ]:
np.linspace(0, 1, 100).shape